# vLLM benchmark

Serves a model from the node's models share and sweeps concurrency levels with
`vllm bench serve`, the way an AI Studio benchmark run does:

* weights come from `/models` (the share, mounted read-only): nothing is downloaded
* vLLM's GPU share is sized to the memory that is free now, less 1.5 GiB headroom
* `--max-num-seqs` is the highest concurrency, so vLLM holds no larger batch
* random prompts of a fixed length, 10 prompts per concurrency level

Run the cells top to bottom. Edit only the configuration cell. Stop the server
at the end to give the GPU back.

## Configuration

In [ ]:
import os

MODEL          = "TinyLlama-1.1B-Chat-v1.0"   # a folder under /models
CONCURRENCIES  = [1, 4, 16]
INPUT_TOKENS   = 512
OUTPUT_TOKENS  = 128
DTYPE          = "float16"     # T4 and older: float16 (bfloat16 needs compute 8.0+)
MAX_MODEL_LEN  = 2048
TENSOR_PARALLEL = 1            # GPUs to split the model across (the notebook's own GPUs)
PORT           = 8010
RESULTS_DIR    = "/tmp/vllm_bench"

MODEL_PATH = os.path.join("/models", MODEL)
print("Models on the share:", ", ".join(sorted(os.listdir("/models"))) if os.path.isdir("/models") else "none (no /models)")
if not os.path.isfile(os.path.join(MODEL_PATH, "config.json")):
    raise SystemExit(f"{MODEL_PATH} is not a model folder. Pick one of the models listed above.")
print("Model:", MODEL_PATH)

## Size vLLM to the free GPU memory

In [ ]:
import math, subprocess

HEADROOM_MIB = 1536   # what vLLM uses outside its share (CUDA graphs, workspace)

def gpu_share(index=0):
    """vLLM's --gpu-memory-utilization for this GPU: its free memory, less the headroom."""
    out = subprocess.check_output(
        ["nvidia-smi", "-i", str(index), "--query-gpu=name,memory.free,memory.total",
         "--format=csv,noheader,nounits"], text=True).strip()
    name, free, total = [p.strip() for p in out.split(",")]
    free, total = int(free), int(total)
    share = min(0.90, math.floor((free - HEADROOM_MIB) / total * 100) / 100)
    print(f"GPU {index}: {name}, {free/1024:.1f} of {total/1024:.1f} GiB free -> share {share:.2f}")
    if share < 0.10:
        raise SystemExit("Too little GPU memory free for vLLM. Launch the notebook on a freer GPU.")
    return share

GPU_SHARE = min(gpu_share(i) for i in range(TENSOR_PARALLEL))

## Start the vLLM server
Loading can take a few minutes. The log is in `/tmp/vllm.log`.

In [ ]:
import subprocess, time, urllib.request

def server_up():
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/v1/models", timeout=2)
        return True
    except Exception:
        return False

if server_up():
    print(f"vLLM is already running on port {PORT}.")
else:
    log = open("/tmp/vllm.log", "w")
    server = subprocess.Popen([
        "vllm", "serve", MODEL_PATH, "--port", str(PORT), "--host", "127.0.0.1",
        "--dtype", DTYPE, "--tensor-parallel-size", str(TENSOR_PARALLEL),
        "--gpu-memory-utilization", f"{GPU_SHARE:.2f}",
        "--max-num-seqs", str(max(CONCURRENCIES)), "--max-model-len", str(MAX_MODEL_LEN),
    ], stdout=log, stderr=subprocess.STDOUT)
    started = time.time()
    while not server_up():
        if server.poll() is not None:
            print(open("/tmp/vllm.log").read()[-4000:])
            raise SystemExit(f"vLLM exited with code {server.returncode}; its last lines are above.")
        time.sleep(5)
        print(f"loading… {int(time.time() - started)}s", end="\r")
    print(f"vLLM ready after {int(time.time() - started)}s on port {PORT}.")

## Run the sweep

In [ ]:
import json, pathlib, shutil

pathlib.Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)
rows = []
for c in CONCURRENCIES:
    name = f"c{c}.json"
    (pathlib.Path(RESULTS_DIR) / name).unlink(missing_ok=True)
    print(f"Concurrency {c} ({10 * c} prompts)…")
    subprocess.run([
        "vllm", "bench", "serve", "--backend", "vllm",
        "--host", "127.0.0.1", "--port", str(PORT), "--endpoint", "/v1/completions",
        "--model", MODEL_PATH, "--dataset-name", "random",
        "--random-input-len", str(INPUT_TOKENS), "--random-output-len", str(OUTPUT_TOKENS),
        "--num-prompts", str(10 * c), "--max-concurrency", str(c), "--ignore-eos",
        "--percentile-metrics", "ttft,tpot,itl,e2el",
        "--save-result", "--result-dir", RESULTS_DIR, "--result-filename", name,
    ], check=True, stdout=subprocess.DEVNULL)
    r = json.load(open(pathlib.Path(RESULTS_DIR) / name))
    rows.append((c, r["total_token_throughput"], r["output_throughput"], r["mean_ttft_ms"],
                 r.get("p99_ttft_ms"), r["mean_tpot_ms"], r["mean_e2el_ms"], r["completed"], r["duration"]))

head = ("Concurrency", "Total tok/s", "Output tok/s", "TTFT mean ms", "TTFT p99 ms",
        "TPOT mean ms", "E2E mean ms", "Requests ok", "Duration s")
print()
print("  ".join(f"{h:>12}" for h in head))
for row in rows:
    print("  ".join(f"{v:>12.2f}" if isinstance(v, float) else f"{v:>12}" for v in row))

## Stop the server
Gives the GPU memory back.

In [ ]:
subprocess.run(["pkill", "-f", f"vllm serve {MODEL_PATH}"])
time.sleep(3)
print("vLLM stopped." if not server_up() else "vLLM is still answering; run this cell again.")